In [1]:
##### Config #####
CONFIG_ROUNDS: int = 7
CONFIG_CLIENTS: int = 5
CONFIG_SHAP_SAMPLE: int = 25
##################

import os
import time
# -------------------------
# Determinism / Reproducibility
# -------------------------
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

import random
random.seed(seed_value)

import numpy as np
np.random.seed(seed_value)

import tensorflow as tf
tf.random.set_seed(seed_value)

try:
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except Exception:
    pass

# -------------------------
# Imports
# -------------------------
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import shap
import matplotlib.pyplot as plt
import warnings
import glob
import pandas as pd
warnings.filterwarnings('ignore')

# =====================================================
# Federated XAI IDS - Model Comparer (MLP vs. Pure CNN)
# =====================================================
class FederatedIDSComparer:
    def __init__(self, num_clients=4, input_dim=46, num_classes=6, batch_size=500000):
        self.num_clients = num_clients
        self.input_dim = input_dim
        self.num_classes = num_classes
        self.batch_size = batch_size
        self.scaler = StandardScaler()
        self.label_encoder = LabelEncoder()

    # -----------------------------------------------------
    # Model Architecture 1: Deep MLP
    # -----------------------------------------------------
    def create_mlp_model(self):
        model = keras.Sequential([
            layers.Dense(256, activation='relu', input_shape=(self.input_dim,)),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(128, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(64, activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(0.3),
            layers.Dense(self.num_classes, activation='softmax')
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    # -----------------------------------------------------
    # Model Architecture 2: Pure 1D CNN Architecture (LSTM বাদ)
    # -----------------------------------------------------
    def create_cnn_model(self):
        model = keras.Sequential([
            layers.Reshape((self.input_dim, 1), input_shape=(self.input_dim,)),
            
            # CNN layers (Feature extraction)
            layers.Conv1D(64, 3, activation='relu', padding='same'),
            layers.MaxPooling1D(pool_size=2),
            layers.Conv1D(128, 3, activation='relu', padding='same'),
            layers.MaxPooling1D(pool_size=2),
            layers.Dropout(0.3),
            
            # Flattening and classification
            layers.Flatten(),
            layers.Dense(128, activation='relu'),
            layers.Dropout(0.5),
            layers.Dense(self.num_classes, activation='softmax')
        ])
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
        return model

    # -----------------------------------------------------
    # Data Processing Pipeline (Task 0-9 CSV file run)
    # -----------------------------------------------------
    def process_files_in_batches(self, folder_path, file_pattern="*.csv", max_files=10, max_samples=None):
        print("Processing files...")
        csv_files = sorted(glob.glob(os.path.join(folder_path, file_pattern)))
        if not csv_files:
            raise ValueError(f"No CSV files found in {folder_path}")
        
        # ০ থেকে ৯ নম্বর পর্যন্ত প্রথম ১০টি ফাইল রিড করবে
        csv_files = csv_files[:max_files]
        print(f"Using {len(csv_files)} files for comparison (parts 0 to 9)")
        
        sample_data = self._get_sample_for_preprocessing(csv_files, sample_size=10000)
        self._fit_preprocessors(sample_data)

        all_client_data = {f'client_{i}': {'X_train': [], 'X_test': [], 'y_train': [], 'y_test': []} 
                          for i in range(self.num_clients)}
        total_processed = 0

        for file_idx, csv_file in enumerate(csv_files):
            print(f"Loading File {file_idx+1}/{len(csv_files)}: {os.path.basename(csv_file)}")
            try:
                for chunk in pd.read_csv(csv_file, chunksize=self.batch_size, low_memory=False, on_bad_lines='skip'):
                    if max_samples and total_processed >= max_samples:
                        break
                    X_processed, y_processed = self._process_chunk(chunk)
                    if X_processed is not None:
                        chunk_client_data = self._split_chunk_among_clients(X_processed, y_processed)
                        for client_id, data in chunk_client_data.items():
                            all_client_data[client_id]['X_train'].append(data['X_train'])
                            all_client_data[client_id]['X_test'].append(data['X_test'])
                            all_client_data[client_id]['y_train'].append(data['y_train'])
                            all_client_data[client_id]['y_test'].append(data['y_test'])
                        total_processed += len(X_processed)
            except Exception as e:
                print(f"Error processing {csv_file}: {e}")
                continue

        final_client_data = []
        for client_id in range(self.num_clients):
            client_key = f'client_{client_id}'
            X_train = np.vstack(all_client_data[client_key]['X_train'])
            X_test = np.vstack(all_client_data[client_key]['X_test'])
            y_train = np.vstack(all_client_data[client_key]['y_train'])
            y_test = np.vstack(all_client_data[client_key]['y_test'])
            final_client_data.append({
                'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test
            })
            print(f"Client {client_id+1}: {len(X_train)} training, {len(X_test)} test samples")
        return final_client_data

    def _get_sample_for_preprocessing(self, csv_files, sample_size=10000):
        sample_data = []
        for csv_file in csv_files:
            try:
                df_sample = pd.read_csv(csv_file, nrows=2000, low_memory=False, on_bad_lines='skip')
                sample_data.append(df_sample)
            except Exception:
                continue
        return pd.concat(sample_data, ignore_index=True)

    def _fit_preprocessors(self, sample_data):
        sample_data = self._clean_data_chunk(sample_data)
        timestamp_cols = [c for c in sample_data.columns if 'time' in c.lower() or 'timestamp' in c.lower()]
        if timestamp_cols:
            sample_data = sample_data.drop(timestamp_cols, axis=1)
        target_col = self._identify_target_column(sample_data)
        X_sample = sample_data.drop(target_col, axis=1)
        y_sample = sample_data[target_col]
        self.input_dim = X_sample.shape[1]
        y_sample = self._convert_to_six_classes(y_sample)
        self.scaler.fit(X_sample)
        self.label_encoder.fit(y_sample)

    def _process_chunk(self, chunk):
        chunk_clean = self._clean_data_chunk(chunk)
        if chunk_clean.empty:
            return None, None
        timestamp_cols = [c for c in chunk_clean.columns if 'time' in c.lower() or 'timestamp' in c.lower()]
        if timestamp_cols:
            chunk_clean = chunk_clean.drop(timestamp_cols, axis=1)
        target_col = self._identify_target_column(chunk_clean)
        X = chunk_clean.drop(target_col, axis=1)
        y = chunk_clean[target_col]
        y = self._convert_to_six_classes(y)
        X_processed = self.scaler.transform(X)
        y_encoded = self.label_encoder.transform(y)
        y_processed = keras.utils.to_categorical(y_encoded, self.num_classes)
        return X_processed, y_processed

    def _clean_data_chunk(self, df_chunk):
        for col in df_chunk.select_dtypes(include=['object']).columns:
            if col not in ['Label', 'label', 'attack', 'class']:
                try:
                    df_chunk[col] = pd.to_numeric(df_chunk[col], errors='coerce')
                except:
                    pass
        df_chunk = df_chunk.replace([np.inf, -np.inf], np.nan)
        df_chunk = df_chunk.dropna()
        return df_chunk

    def _identify_target_column(self, df):
        for c in ['label', 'Label', 'attack', 'Attack', 'class', 'Class']:
            if c in df.columns:
                return c
        return df.columns[-1]

    def _split_chunk_among_clients(self, X_chunk, y_chunk):
        chunk_size = len(X_chunk)
        samples_per_client = chunk_size // self.num_clients
        client_data = {}
        for i in range(self.num_clients):
            start_idx, end_idx = i * samples_per_client, (i + 1) * samples_per_client
            if i == self.num_clients - 1:
                end_idx = chunk_size
            X_client, y_client = X_chunk[start_idx:end_idx], y_chunk[start_idx:end_idx]
            X_train, X_test, y_train, y_test = train_test_split(
                X_client, y_client, test_size=0.2, random_state=seed_value
            )
            client_data[f'client_{i}'] = {
                'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test
            }
        return client_data

    def _convert_to_six_classes(self, y):
        y_str = y.astype(str).str.lower()
        class_mapping = {}
        for lbl in y_str.unique():
            lbl_low = lbl.lower()
            if any(k in lbl_low for k in ['benign', 'normal', 'legitimate']):
                class_mapping[lbl] = 'Benign'
            elif any(k in lbl_low for k in ['ddos', 'distributed denial']):
                class_mapping[lbl] = 'DDoS'
            elif any(k in lbl_low for k in ['dos', 'denial of service']):
                class_mapping[lbl] = 'DoS'
            elif 'mirai' in lbl_low:
                class_mapping[lbl] = 'Mirai'
            elif any(k in lbl_low for k in ['recon', 'scan', 'portscan', 'hostscan']):
                class_mapping[lbl] = 'Recon'
            else:
                class_mapping[lbl] = 'Other'
        return y_str.map(class_mapping)

    # -----------------------------------------------------
    # Federated Training Pipeline (With Encryption Placeholders)
    # -----------------------------------------------------
    def train_federated_model(self, client_data, model_type="mlp", rounds=5, epochs=10):
        print(f"\nTraining Federated {model_type.upper()} Model...")
        if model_type == "mlp":
            global_model = self.create_mlp_model()
        else:
            global_model = self.create_cnn_model()

        global_weights = global_model.get_weights()
        round_accuracies = []

        for r in range(rounds):
            client_weights, client_samples = [], []
            for i, data in enumerate(client_data):
                if len(data['X_train']) == 0:
                    continue
                
                if model_type == "mlp":
                    local_model = self.create_mlp_model()
                else:
                    local_model = self.create_cnn_model()
                
                local_model.set_weights(global_weights)
                local_model.fit(
                    data['X_train'], data['y_train'],
                    epochs=epochs, batch_size=256, verbose=0
                )
                
                # --------------------------------------------------
                # 🔒 SECURITY PLACEHOLDER (Homomorphic Encryption + AES)
                # --------------------------------------------------
                # Aggregation এর পূর্বে লোকাল ক্লায়েন্ট ওয়েটস AES দিয়ে এনক্রিপ্ট হবে 
                # অথবা Homomorphic Encryption ব্যবহার করে ডিক্রিপ্ট না করেই যোগ হবে।
                # --------------------------------------------------
                client_weights.append(local_model.get_weights())
                client_samples.append(len(data['X_train']))

            # FedAvg Weight Aggregation
            if client_weights:
                new_global_weights = []
                total_samples = sum(client_samples)
                for weights in zip(*client_weights):
                    agg = np.zeros_like(weights[0])
                    for w, n in zip(weights, client_samples):
                        agg += w * (n / total_samples)
                    new_global_weights.append(agg)
                global_weights = new_global_weights
                global_model.set_weights(global_weights)

            # Evaluate at the end of the round
            X_test_all = np.vstack([d['X_test'] for d in client_data if len(d['X_test']) > 0])
            y_test_all = np.vstack([d['y_test'] for d in client_data if len(d['y_test']) > 0])
            _, acc = global_model.evaluate(X_test_all, y_test_all, verbose=0)
            round_accuracies.append(acc)
            print(f"  Round {r+1}/{rounds} Global Accuracy: {acc:.4f}")

        return global_model, round_accuracies

    # -----------------------------------------------------
    # Generate SHAP explanations
    # -----------------------------------------------------
    def apply_shap_explainability(self, model, X_test, model_name="model"):
        print(f"\n📊 Applying Kernel SHAP for {model_name.upper()} Explainability...")
        try:
            sample_size = min(CONFIG_SHAP_SAMPLE, X_test.shape[0])
            rng = np.random.default_rng(seed_value)
            shap_idx = rng.choice(X_test.shape[0], sample_size, replace=False)
            X_shap = X_test[shap_idx]
            
            background_size = min(100, X_test.shape[0])
            background_idx = rng.choice(X_test.shape[0], background_size, replace=False)
            X_bg = X_test[background_idx]
            
            explainer = shap.KernelExplainer(model.predict, X_bg)
            shap_vals = explainer.shap_values(X_shap, nsamples=100)
            
            # Save Bar plot
            plt.figure()
            shap.summary_plot(shap_vals, X_shap, show=False, plot_type="bar")
            plt.title(f"Global Feature Importance ({model_name.upper()})")
            plt.tight_layout()
            plt.savefig(f"shap_{model_name}_feature_importance_bar.png")
            plt.close()
            
            # Save Dot Summary plot
            plt.figure()
            shap.summary_plot(shap_vals, X_shap, show=False)
            plt.title(f"Global SHAP Summary ({model_name.upper()})")
            plt.tight_layout()
            plt.savefig(f"shap_{model_name}_summary_dot.png")
            plt.close()
            print(f"  SHAP plots saved successfully as 'shap_{model_name}_*.png'")
        except Exception as e:
            print(f"  SHAP explanation failed for {model_name}: {e}")

# -----------------------------------------------------
# Detailed Evaluation (Precision, Recall, Support, Confusion Matrix)
# -----------------------------------------------------
def evaluate_and_plot_metrics(y_true, y_pred_probs, classes, model_name="model"):
    y_pred = np.argmax(y_pred_probs, axis=1)
    
    # Precision, Recall, Support, and F1-Score
    print(f"\n==================================================")
    print(f"📊 EVALUATION METRICS FOR {model_name.upper()} (Precision, Recall, Support)")
    print(f"==================================================")
    print(classification_report(y_true, y_pred, target_names=classes))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(8, 6))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
    disp.plot(cmap=plt.cm.Blues, values_format='d', ax=ax, xticks_rotation=45)
    plt.title(f'Confusion Matrix: {model_name.upper()}')
    plt.tight_layout()
    plt.savefig(f'{model_name}_confusion_matrix.png', dpi=300)
    plt.close()
    print(f"📁 Confusion Matrix saved successfully as '{model_name}_confusion_matrix.png'\n")

# -----------------------------------------------------
# 📊 Grouped Bar Chart Visual (১ম ছবির মতো হুবহু গ্রাফ তৈরির ফাংশন)
# -----------------------------------------------------
def plot_grouped_metrics_chart(report_dict, classes, model_name="model"):
    # Precision, Recall এবং F1-Score এক্সট্র্যাক্ট করা
    precision = [report_dict[cls]['precision'] for cls in classes]
    recall = [report_dict[cls]['recall'] for cls in classes]
    f1_score = [report_dict[cls]['f1-score'] for cls in classes]

    x = np.arange(len(classes))
    width = 0.25  # প্রতিটি বারের প্রস্থ

    fig, ax = plt.subplots(figsize=(12, 7))
    
    # ১ম ছবির রঙের থিম অনুযায়ী বারগুলো প্লট করা
    rects1 = ax.bar(x - width, precision, width, label='Precision', color='#5c7cb8')
    rects2 = ax.bar(x, recall, width, label='Recall', color='#d68a5c')
    rects3 = ax.bar(x + width, f1_score, width, label='F1-Score', color='#70c270')

    # গ্রাফ সাজানো
    ax.set_ylabel('Score')
    ax.set_xlabel('Class Label')
    ax.set_title(f'Classification Performance Metrics per Class (Risk Analysis - {model_name.upper()})')
    ax.set_xticks(x)
    ax.set_xticklabels(classes, rotation=45)
    ax.set_ylim(0, 1.1)
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    ax.legend(title="Metric", loc="lower right")

    plt.tight_layout()
    plt.savefig(f'{model_name}_class_performance_metrics.png', dpi=300)
    plt.close()
    print(f"📁 Grouped Metrics Bar Chart saved as '{model_name}_class_performance_metrics.png'")


# =====================================================
# Main Evaluation Pipeline (Deep MLP vs. Pure 1D-CNN)
# =====================================================
if __name__ == "__main__":
    comparer = FederatedIDSComparer(num_clients=CONFIG_CLIENTS, input_dim=46, num_classes=6, batch_size=500000)
    data_folder = r"CICIoT2023"
    file_pattern = "*.csv"

    try:
        print("==============================================================")
        print("=== COMPARING DEEP MLP VS. PURE 1D-CNN UNDER FEDERATED ======")
        print("==============================================================")

        # Step 1: Process Dataset once for both models (0 to 9 CSV parts = max_files=10)
        client_data = comparer.process_files_in_batches(data_folder, file_pattern, max_files=10)
        X_test_all = np.vstack([d['X_test'] for d in client_data])
        y_test_all = np.vstack([d['y_test'] for d in client_data])
        y_true_classes = np.argmax(y_test_all, axis=1)

        # Step 2: Train & Evaluate Deep MLP
        mlp_start = time.time()
        mlp_model, mlp_acc_history = comparer.train_federated_model(
            client_data, model_type="mlp", rounds=CONFIG_ROUNDS, epochs=10
        )
        mlp_time = time.time() - mlp_start
        mlp_preds = mlp_model.predict(X_test_all, verbose=0)

        # Step 3: Train & Evaluate Pure 1D-CNN (LSTM বাদ)
        cnn_start = time.time()
        cnn_model, cnn_acc_history = comparer.train_federated_model(
            client_data, model_type="cnn", rounds=CONFIG_ROUNDS, epochs=10
        )
        cnn_time = time.time() - cnn_start
        cnn_preds = cnn_model.predict(X_test_all, verbose=0)

        # Save both trained global models
        mlp_model.save("fedxaiids_MLP_TEST.h5")
        cnn_model.save("fedxaiids_CNN_TEST.h5")
        print("\nBoth models saved successfully.")

        # Save preprocessing artifacts
        import joblib
        joblib.dump(comparer.scaler, 'scaler_TEST.pkl')
        joblib.dump(comparer.label_encoder, 'label_encoder_TEST.pkl')
        print("Preprocessing objects saved.")

        # =====================================================
        # COMPARATIVE PLOTTING & VISUALIZATION
        # =====================================================
        print("\n📉 Generating Comparative Visualizations...")
        classes = list(comparer.label_encoder.classes_)
        rounds_range = range(1, CONFIG_ROUNDS + 1)

        # ১. ডিকশনারি আকারে রিপোর্ট জেনারেট করা (গ্রুপড বার চার্ট তৈরির জন্য)
        mlp_pred_classes = np.argmax(mlp_preds, axis=1)
        cnn_pred_classes = np.argmax(cnn_preds, axis=1)
        
        mlp_report_dict = classification_report(y_true_classes, mlp_pred_classes, target_names=classes, output_dict=True)
        cnn_report_dict = classification_report(y_true_classes, cnn_pred_classes, target_names=classes, output_dict=True)

        # ২. গ্রুপড বার চার্ট জেনারেট করা (হুবহু ১ম ছবির গ্রাফ)
        plot_grouped_metrics_chart(mlp_report_dict, classes, model_name="mlp")
        plot_grouped_metrics_chart(cnn_report_dict, classes, model_name="cnn")

        # ৩. কনফিউশন ম্যাট্রিক্স এবং বিশদ বিবরণ প্রিন্ট করা
        evaluate_and_plot_metrics(y_true_classes, mlp_preds, classes, model_name="mlp")
        evaluate_and_plot_metrics(y_true_classes, cnn_preds, classes, model_name="cnn")

        # Graph 1: Convergence Trend (Accuracy vs Rounds)
        plt.figure(figsize=(8, 5))
        plt.plot(rounds_range, mlp_acc_history, marker='o', label='Deep MLP', color='#1f77b4', linewidth=2)
        plt.plot(rounds_range, cnn_acc_history, marker='s', label='Pure 1D-CNN', color='#d62728', linewidth=2)
        plt.title('Global Model Accuracy vs. Federated Rounds')
        plt.xlabel('Federated Round')
        plt.ylabel('Global Accuracy')
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.savefig('comparison_rounds_accuracy.png', dpi=300)
        plt.close()

        # Graph 2: Class-wise F1-Score Comparison (দুইটি মডেলের F1-Score তুলনা)
        mlp_f1s = [mlp_report_dict[cls]['f1-score'] for cls in classes]
        cnn_f1s = [cnn_report_dict[cls]['f1-score'] for cls in classes]
        
        x = np.arange(len(classes))
        width = 0.35
        
        plt.figure(figsize=(10, 6))
        plt.bar(x - width/2, mlp_f1s, width, label='Deep MLP', color='#90caf9')
        plt.bar(x + width/2, cnn_f1s, width, label='Pure 1D-CNN', color='#ef9a9a')
        plt.title('Class-wise F1-Score Comparison')
        plt.xlabel('Security Attack Class')
        plt.ylabel('F1-Score')
        plt.xticks(x, classes)
        plt.ylim(0, 1.15)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.legend(loc='upper right')
        plt.tight_layout()
        plt.savefig('comparison_class_f1.png', dpi=300)
        plt.close()

        # Graph 3: Computation Training Time Comparison
        plt.figure(figsize=(6, 5))
        models = ['Deep MLP', 'Pure 1D-CNN']
        times = [mlp_time, cnn_time]
        colors = ['#90caf9', '#ef9a9a']
        bars = plt.bar(models, times, color=colors, width=0.45)
        plt.title('Total Federated Training Latency')
        plt.ylabel('Seconds (s)')
        for bar in bars:
            height = bar.get_height()
            plt.text(bar.get_x() + bar.get_width()/2.0, height + (max(times)*0.01), 
                     f'{height:.1f}s', ha='center', va='bottom', fontweight='bold')
        plt.tight_layout()
        plt.savefig('comparison_training_time.png', dpi=300)
        plt.close()

        # Step 4: Apply SHAP independently for both models (২য় ছবির SHAP গ্রাফ তৈরির জন্য)
        comparer.apply_shap_explainability(mlp_model, X_test_all, model_name="mlp")
        comparer.apply_shap_explainability(cnn_model, X_test_all, model_name="cnn")

        print("\n" + "="*50)
        print("🎉 ALL COMPARISONS AND PLOTS GENERATED SUCCESSFULLY!")
        print("Generated files:")
        print(" - comparison_rounds_accuracy.png")
        print(" - comparison_class_f1.png")
        print(" - comparison_training_time.png")
        print(" - mlp_class_performance_metrics.png & cnn_class_performance_metrics.png (১ম ছবির গ্রাফ)")
        print(" - mlp_confusion_matrix.png & cnn_confusion_matrix.png")
        print(" - shap_mlp_*.png & shap_cnn_*.png (২য় ছবির গ্রাফ)")
        print("="*50)

    except Exception as e:
        print(f"Error during comparison pipeline: {e}")
        import traceback
        traceback.print_exc() 

c:\Users\Mehedi\anaconda4\envs\tf-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== COMPARING DEEP MLP VS. PURE 1D-CNN UNDER FEDERATED ======
Processing files...
Using 10 files for comparison (parts 0 to 9)
Loading File 1/10: part-00001-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 2/10: part-00002-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 3/10: part-00003-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 4/10: part-00004-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 5/10: part-00005-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 6/10: part-00006-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 7/10: part-00007-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 8/10: part-00008-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 9/10: part-00009-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Loading File 10/10: part-00010-363d1ba3-8ab5-4f96-bc25-4d5862db7cb9-c000.csv
Client 1: 380739 training, 95190 test samples
Client 2: 380739 training, 95190 test samples
Client 3: 380739 tra

  0%|          | 0/25 [00:00<?, ?it/s]

313/313 [==============================] - 1s 2ms/step


  4%|▍         | 1/25 [00:01<00:29,  1.23s/it]

...

313/313 [==============================] - 1s 2ms/step


100%|██████████| 25/25 [00:26<00:00,  1.07s/it]


  SHAP plots saved successfully as 'shap_cnn_*.png'

🎉 ALL COMPARISONS AND PLOTS GENERATED SUCCESSFULLY!
Generated files:
 - comparison_rounds_accuracy.png
 - comparison_class_f1.png
 - comparison_training_time.png
 - mlp_class_performance_metrics.png & cnn_class_performance_metrics.png (১ম ছবির গ্রাফ)
 - mlp_confusion_matrix.png & cnn_confusion_matrix.png
 - shap_mlp_*.png & shap_cnn_*.png (২য় ছবির গ্রাফ)


<Figure size 640x480 with 0 Axes>

<Figure size 640x480 with 0 Axes>